# Instructions

This notebook will generate a readout of KM metadata (records, KPIs, filters, etc.) as well as View and KM variables (from views with the same KM)
Steps for setup and execution:
1. Fill out the User Input Section.
    - Input the Space ID (found in url after 'space/'), Package ID (found in url after 'package/') and the KM ID (found in url after 'node/')
2. Restart Kernel and Run All Cells (>> button at the top)

Notes:
- There are a lot of different attributes for each KM component (disply name, desc, pql, etc) - I am extracting what I believe to be relevant, but there may be attributes of some components that you are looking for that are missing. If so, slack me (Andrew Cirincione) and I will try to add it in!

# User Input

In [1]:
# Import environment and workspace variables
team_url = '' # URL of your team (ex: https://xxx.xxx.celonis.cloud/)
api_key = '' # API key
key_type = 'USER_KEY' # Can either be USER_KEY or APP_KEY
space_id = ''
package_id = ''
km_id = ''

# Import Packages

In [2]:
!pip install --extra-index-url=https://pypi.celonis.cloud/ pycelonis --upgrade
!pip install xlsxwriter
# !pip install business_landscape_pycelonis --extra-index-url https://pypi.celonis.cloud/c591f5a8f4874916a268b4911580bebb --extra-index-url https://pypi.celonis.cloud/

from pycelonis import get_celonis
import pandas as pd
import xlsxwriter
from tqdm import tqdm
import json

Looking in indexes: https://pypi.celonis.cloud, https://pypi.org/simple, https://pypi.celonis.cloud/
  Using cached https://pypi.celonis.cloud/saolapy/saolapy-0.3.3-py3-none-any.whl (41 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
  Attempting uninstall: h11
    Found existing installation: h11 0.14.0
    Uninstalling h11-0.14.0:
      Successfully uninstalled h11-0.14.0
  Attempting uninstall: saolapy
    Found existing installation: saolapy 0.3.0
    Uninstalling saolapy-0.3.0:
      Successfully uninstalled saolapy-0.3.0
  Attempting uninstall: httpcore
    Found existing installation: httpcore 0.17.3
    Uninstalling httpcore-0.17.3:
      Successfully uninstalled httpcore-0.17.3
  Attem

# Load Celonis

In [3]:
celonis = get_celonis(team_url, api_key, key_type)

[2026-03-25 14:16:30,574] INFO: Initial connect successful! PyCelonis Version: 2.14.2
[2026-03-25 14:16:30,843] INFO: `ml-workbench` permissions: ['CREATE_APPS', 'USE_ALL_APPS', 'MANAGE_ALL_APPS', 'CREATE_WORKSPACES', 'MANAGE_ALL_WORKSPACES', 'VIEW_CONFIGURATION']
[2026-03-25 14:16:30,844] INFO: `team` permissions: ['MANAGE_AUDIT_LOGS', 'MANAGE_SSO_SETTINGS', 'USE_AUDIT_LOGS_API', 'TEAM_TO_TEAM_COPY', 'MANAGE_ADOPTION_VIEWS', 'MANAGE_GENERAL_SETTINGS', 'MANAGE_GROUPS', 'MANAGE_APPLICATIONS', 'MANAGE_ON_PREM_CLIENTS', 'USE_STUDIO_ADOPTION_API', 'MANAGE_LOGIN_HISTORY', 'MANAGE_LICENSE_SETTINGS', 'USE_LOGIN_HISTORY_API', 'USE_USER_GROUP_INFO_API', 'MANAGE_MEMBERS', 'MANAGE_UPLINK_INTEGRATIONS', 'MANAGE_PERMISSIONS', 'MANAGE_ADMIN_NOTIFICATIONS', 'MANAGE_DOWNLOAD_PORTAL', 'IMPORT_MEMBERS']
[2026-03-25 14:16:30,844] INFO: `storage-manager` permissions: ['DELETE', 'CREATE', 'GET', 'ADMIN', 'LIST']
[2026-03-25 14:16:30,845] INFO: `package-manager` permissions: ['EXPORT_ALL', 'ENABLE_AI_ML_ASS

In [7]:
celonis.data_integration

# Load Space, Package, and KM

In [9]:
space = celonis.studio.get_space(space_id)
package = space.get_package(package_id)
km = package.get_knowledge_model(km_id)
km_serialized_content = json.loads(km.serialized_content)
for k in package.get_knowledge_models():
    if k.key == km_serialized_content['metadata']['key']:
        km_root_node_key = k.root_node_key 
        km_key = k.key
payload = {'withAutogeneratedDataModelData': True
           ,'withResolvedScopes': False
           ,'withTeamAndCelonisKnowledge': False
           ,'withVariableReplacement': True}
km_content = json.loads(celonis.client.request(url=f'{team_url}semantic-layer/api/layer/{km_root_node_key}.{km_key}/final', method = 'POST', json=payload).content)

# Load KM Metadata

In [11]:
el_dict = None
kpi_dict = None
aa_dict = None
filter_dict = None
trigger_dict = None
action_dict = None
flag_dict = None
var_dict = None
legacy_el_dict = None
view_var_df = None
km_var_df = None
record_df = None 

# Records
record_rows = []
for record in km_content['layer']['records']:

    for attribute in record['attributes']:
        record_rows.append({
            'RECORD_ID': record['id'],
            'RECORD_DISPLAY_NAME': record['displayName'],
            'RECORD_SHORT_DISPLAY_NAME': record['shortDisplayName'],
            'RECORD_DESCRIPTION': record['description'],
            'RECORD_PQL': record['pql'],
            'RECORD_AUTOGENERATED': record['autoGenerated'],
            'ATTRIBUTE_ID': attribute['id'],
            'ATTRIBUTE_DISPLAY_NAME': attribute['displayName'],
            'ATTRIBUTE_SHORT_DISPLAY_NAME': attribute['shortDisplayName'],
            'ATTRIBUTE_DESCRIPTION': attribute['description'],
            'ATTRIBUTE_PQL': attribute['pql'],
            'ATTRIBUTE_COLUMN_TYPE': attribute['columnType'],
            'ATTRIBUTE_UNIT': attribute['unit'],
            'ATTRIBUTE_FORMAT': attribute['format'],
            'ATTRIBUTE_SOURCE': attribute['attributeSource'],
            'ATTRIBUTE_AUTOGENERATED': attribute['autoGenerated'],
        })
record_df = pd.DataFrame(record_rows)

# Triggers
trigger_flag = 0
if 'triggers' in km_content['layer'].keys():
    trigger_flag = 1
    trigger_rows = []
    for trigger in km_content['layer']['triggers']:

        for attribute in trigger['columns']:
            trigger_rows.append({
                'TRIGGER_ID': trigger['id'],
                'TRIGGER_DISPLAY_NAME': trigger['displayName'],
                'TRIGGER_SHORT_DISPLAY_NAME': trigger['shortDisplayName'],
                'TRIGGER_DESCRIPTION': trigger['description'],
                'TRIGGER_TABLE_NAME': trigger['tableName'],
                'TRIGGER_FILTERS': trigger['filters'],
                'TRIGGER_AUTOGENERATED': trigger['autoGenerated'],
                'COLUMN_ID': attribute['id'],
                'COLUMN_DISPLAY_NAME': attribute['displayName'],
                'COLUMN_SHORT_DISPLAY_NAME': attribute['shortDisplayName'],
                'COLUMN_DESCRIPTION': attribute['description'],
                'COLUMN_PQL': attribute['pql'],
                'COLUMN_COLUMN_TYPE': attribute['columnType'],
                'COLUMN_UNIT': attribute['unit'],
                'COLUMN_FORMAT': attribute['format'],
                'COLUMN_IDENTIFIER': attribute['identifier'],
                'COLUMN_AUTOGENERATED': attribute['autoGenerated'],
            })
    trigger_df = pd.DataFrame(trigger_rows)
    
# KPIs
kpi_dict = {}
for kpi in km_content['layer']['kpis']:
    kpi_dict[kpi['id']] = {'ID': kpi['id']
                           ,'DISPLAY_NAME': kpi['displayName']
                           ,'SHORT_DISPLAY_NAME': kpi['shortDisplayName']
                           ,'DESCRIPTION': kpi['description']
                           ,'PQL': kpi['pql']
                           ,'UNIT': attribute['unit']
                           ,'FORMAT': attribute['format']
                           ,'AUTOGENERATED': kpi['autoGenerated']
                          }

# Variables
var_dict = {}
for var in km_content['layer']['variables']:
    var_dict[var['id']] = {'ID': var['id']
                           ,'DISPLAY_NAME': var['displayName']
                           ,'SHORT_DISPLAY_NAME': var['shortDisplayName']
                           ,'DESCRIPTION': var['description']
                           ,'VALUE': var['value']
                          }
# Augmented Attributes
aa_flag = 0
if 'augmentedAttributes' in km_content['layer'].keys():
    aa_flag = 1
    aa_dict = {}
    for aa in km_content['layer']['augmentedAttributes']:
        aa_dict[aa['id']] = {'ID': aa['id']
                               ,'DISPLAY_NAME': aa['displayName']
                               ,'SHORT_DISPLAY_NAME': aa['shortDisplayName']
                               ,'DESCRIPTION': aa['description']
                               ,'POSSIBLE_VALUES': aa['possibleValues']
                               ,'DEFAULT_VALUE': aa['defaultValue']
                               ,'DATA_MODEL_TABLE': aa['dataModelTable']
                               ,'AUGMENTATION_TABLE_NAME_SUFFIX': aa['augmentationTableNameSuffix']
                               ,'PQL': aa['pql']
                               ,'HOUSEKEEPING': aa['housekeeping']
                               ,'AUTOGENERATED': aa['autoGenerated']
                              }
# Filters
filter_dict = {}
for fil in km_content['layer']['filters']:
    filter_dict[fil['id']] = {'ID':fil['id']
                            ,'DISPLAY_NAME': fil['displayName']
                            ,'SHORT_DISPLAY_NAME': fil['shortDisplayName']
                            ,'DESCRIPTION': fil['description']
                            ,'PQL': fil['pql']
                            ,'GLOBAL': fil['global']
                          }
# Event Logs (legacy)
el_legacy_dict = {}
for el_legacy in km_content['layer']['eventLogsMetadata']['eventLogs']:
    el_legacy_dict[el_legacy['id']] = {'ID': el_legacy['id']
                         ,'DISPLAY_NAME': el_legacy['displayName']
                         ,'SHORT_DISPLAY_NAME': el_legacy['shortDisplayName']
                         ,'DESCRIPTION': el_legacy['description']
                         ,'PQL': el_legacy['pql']
                         ,'AUTOGENERATED': el_legacy['autoGenerated']
                         ,'DEFAULT_EVENT_LOG': el_legacy['defaultEventLog']
                          }
    
# Actions
action_flag = 0
if 'actions' in km_content['layer'].keys():
    action_flag = 1
    action_dict = {}
    for ac in km_content['layer']['actions']:
        action_dict[ac['id']] = {'ID': ac['id']
                             ,'DISPLAY_NAME': ac['displayName']
                             ,'SHORT_DISPLAY_NAME': ac['shortDisplayName']
                             ,'DESCRIPTION': ac['description']
                             ,'SKILL_ID': ac['skillId']
                             ,'SKILL_KEY': ac['skillKey']
                             ,'RECORDS': ac['records']
                             ,'FILTERS': ac['filters']
                             ,'INPUTS': ac['inputs']
                             ,'EXECUTE_ACTION_FLOW': ac['executeActionFlow']
                             ,'AUTOGENERATED': ac['autoGenerated']
                              }
    
# Flags
flag_dict = {}
for record in km_content['layer']['records']:
        for flag in record['flags']:
            flag_dict[flag['id']] = {'ID': flag['id']
                         ,'DISPLAY_NAME': flag['displayName']
                         ,'SHORT_DISPLAY_NAME': flag['shortDisplayName']
                         ,'DESCRIPTION': flag['description']
                         ,'PQL': flag['pql']
                         ,'RECORD_ID': record['id']
                          }
            
# Event Logs
el_flag = 0
if type(km_content['layer']['eventLogsV2']) != type(None):
    el_flag = 1
    el_dict = {}
    for el in km_content['layer']['eventLogsV2']:
        el_dict[el['id']] = {'ID': el['id']
                             ,'DISPLAY_NAME': el['displayName']
                             ,'SHORT_DISPLAY_NAME': el['shortDisplayName']
                             ,'DESCRIPTION': el['description']
                             ,'LEAD_OBJECT_ID': el['leadObjectId']
                             ,'OBJECT_FILTER': el['objectFilter']
                             ,'EVENT_TYPES': el['eventTypes']
                             ,'ACTIVITY_COLUMN_ID': el['activityColumnId']
                             ,'SORTING_COLUMN': el['sortingColumn']
                             ,'ACTIVITY_COLUMN_PQL':el['activityColumnPql']
                             ,'REGISTERED_EVENT_LOG_TABLE_NAME': el['registeredEventLogTableName']
                             ,'REGISTERED_EDGE_TABLE_NAME': el['registeredEdgeTableName']
                              }

# Load Relevant View and KM Variables

In [12]:
view_var_content = {}
view_var_rows = []
km_var = []
for view in tqdm(package.get_views()):
    view_content = json.loads(celonis.client.request(url = f'{team_url}/blueprint/api/v2/boards/by-key/{view.key}/initial-load-v2?rootNodeKey={view.root_node_key}', method='GET').content)
    if view_content['finalBoard']['configuration']['metadata']['knowledgeModelKey'] == km_key:
        view_var_content[view.key] = view_content['viewVariables']
        km_var = view_content['kmVariables']
        for var in view_var_content[view.key]:
            view_var_rows.append({
                'VIEW_NAME': view.name
                ,'VIEW_KEY': view.root_node_key
                ,'VARIABLE_KEY': var["key"]
                ,'VARIABLE_TYPE': var["dataType"]
                ,'VARIABLE_DESCRIPTION': var["description"]
                ,'VARIABLE_DEFAULT_VALUE': var["defaultValue"]
            })
        
km_var_rows = []
for var in km_var:
    km_var_rows.append({'VARIABLE_KEY': var["key"]
            ,'VARIABLE_TYPE': var["dataType"]
            ,'VARIABLE_DESCRIPTION': var["description"]
            ,'VARIABLE_DEFAULT_VALUE': var["defaultValue"]})
        
view_var_df = pd.DataFrame(view_var_rows)
km_var_df = pd.DataFrame(km_var_rows)

100%|██████████| 24/24 [00:19<00:00,  1.26it/s]


# Generate Output

In [13]:
write_items = [
    ("Event Logs", el_dict),
    ("Records", record_df),
    ("KPIs", kpi_dict),
    ("Augmented Attributes", aa_dict),
    ("Filters", filter_dict),
    ("Triggers", trigger_dict),
    ("Actions", action_dict),
    ("Flags", flag_dict),
    ("Variables (legacy)", var_dict),
    ("Event Logs (legacy)", el_legacy_dict),
    ("View Variables", view_var_df),
    ("KM Variables", km_var_df),
]
with pd.ExcelWriter('KM_Readout_FINAL.xlsx',engine='xlsxwriter') as writer:
    for sheet_name, obj in write_items:

        if obj is None:
            continue

        if isinstance(obj, dict):
            if not obj:
                continue
            df = pd.DataFrame.from_dict(obj, orient='index')

        elif isinstance(obj, pd.DataFrame):
            if obj.empty:
                continue
            df = obj

        else:
            continue

        df.to_excel(writer, sheet_name=sheet_name, index=False)